Validação dos fluxos turísticos

Brasil ↔ Coreia do Sul · 1998–2025

--------------------

Se todos os anos possuem 12 meses para cada fluxo.

Se a soma das chegadas mensais corresponde ao total anual.

Se existem divergências ou registros incompletos.

Se os indicadores estão prontos para as análises posteriores.

In [0]:
from pyspark.sql import functions as F

df_anual = spark.table("chegadas.gold.turismo_bilateral_anual")
df_mensal = spark.table("chegadas.gold.turismo_bilateral_mensal")

In [0]:
df_soma_mensal = (
    df_mensal

    .groupBy("ano", "origem", "destino")

    .agg(
        F.sum("chegadas_observadas").alias(
            "soma_meses_observados"
        ),

        F.countDistinct("mes").alias(
            "meses_disponiveis"
        ),

        F.sum("registros_nulos").alias(
            "registros_nulos_mensais"
        )
    )
)

display(
    df_soma_mensal
    .orderBy("ano", "origem")
)

In [0]:
df_validacao = (
    df_anual.alias("a")

    .join(
        df_soma_mensal.alias("m"),
        on=["ano", "origem", "destino"],
        how="full"
    )

    .select(
        "ano",
        "origem",
        "destino",

        F.col("a.chegadas_observadas").alias(
            "total_anual"
        ),

        "soma_meses_observados",
        "meses_disponiveis",
        "registros_nulos_mensais"
    )

    .withColumn(
        "diferenca",
        F.col("total_anual") -
        F.col("soma_meses_observados")
    )

    .orderBy("ano", "origem")
)

display(df_validacao)

In [0]:
df_validacao = (
    df_validacao

    .withColumn(
        "status_validacao",

        F.when(
            F.col("total_anual").isNull() |
            F.col("soma_meses_observados").isNull(),
            "dados_ausentes"
        )

        .when(
            F.col("meses_disponiveis") != 12,
            "meses_incompletos"
        )

        .when(
            F.col("diferenca") != 0,
            "divergente"
        )

        .when(
            F.col("registros_nulos_mensais") > 0,
            "coincidente_com_nulos"
        )

        .otherwise("coincidente")
    )
)

display(
    df_validacao
    .groupBy("status_validacao")
    .count()
    .orderBy("status_validacao")
)

In [0]:
display(
    df_validacao
    .filter(
        F.col("status_validacao") != "coincidente"
    )
    .orderBy("ano", "origem")
)

In [0]:
df_validacao_final = (
    df_validacao
    .filter(F.col("ano").between(1998, 2025))
)

display(
    df_validacao_final
    .groupBy("status_validacao")
    .count()
    .orderBy("status_validacao")
)

In [0]:
assert df_validacao_final.count() == 56, (
    "Quantidade inesperada de comparações."
)

assert (
    df_validacao_final
    .filter(
        F.col("status_validacao") != "coincidente"
    )
    .filter(
        F.col("status_validacao") != "coincidente_com_nulos"
    )
    .count()
) == 0, "Existem divergências ou dados ausentes."

print("Validação cruzada concluída!")


## Resultado da validação cruzada

Foi realizada a comparação entre os totais anuais e a soma
das chegadas mensais dos dois fluxos turísticos.

Período analisado: 1998–2025.

Foram verificadas 56 combinações de ano e fluxo turístico.

Resultados:
- 53 comparações apresentaram coincidência numérica sem
  valores nulos nos registros mensais.
- 3 comparações apresentaram coincidência numérica, mas
  continham registros com valores nulos.
- Nenhuma divergência numérica foi identificada.

Os registros com valores nulos pertencem ao fluxo
Coreia do Sul → Brasil, nos anos de 1999, 2012 e 2014.

Esses totais representam a soma dos valores disponíveis
e não devem ser interpretados como necessariamente completos.

O ano de 1996 também apresenta registros nulos na base
brasileira, mas está fora do período de comparação mensal.

Os anos de 1990 a 1997 foram excluídos somente da validação
cruzada, pois não fazem parte do período mensal comum.

As tabelas Gold anual e mensal permanecem preservadas.